In [1]:
import argparse
import os
import pickle
import time

from importlib import metadata
import torch
try:
    try:
        if metadata.version("rsl-rl"):
            raise ImportError
    except metadata.PackageNotFoundError:
        if metadata.version("rsl-rl-lib") != "3.1.1":  #2.2.4
            raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please uninstall 'rsl_rl' and install 'rsl-rl-lib==2.2.4'.") from e
from rsl_rl.runners import OnPolicyRunner

In [2]:
from bp000_env_cnoid import BP000Env as RLEnv

In [3]:
# 任意設定項目
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-ridho-model'  # ckpt = 4000
exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-22'
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-9'  # min_ankle_height 弊害
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-8'  #  暫定１位
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-6'
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-5'
ckpt = 100

action_scale = 1.0 # 動作のスケールを調整

In [4]:
# 既存のセルを置き換え
import pandas as pd
import numpy as np

# データ収集用のリスト
# action_data = []
obs_data = []
torque_data = []
step_data = []

# CSVファイルの準備
csv_filename = f'obs_data/{exp_name}_step_data.csv'
os.makedirs('obs_data', exist_ok=True)

In [5]:
def _obs_vec(obs):
    # TensorDict or dict → 'policy' を優先
    if isinstance(obs, dict) or hasattr(obs, "get"):
        if "policy" in obs:
            obs = obs["policy"]
    if torch.is_tensor(obs):
        return obs.detach().cpu().numpy().ravel()
    return np.asarray(obs, dtype=np.float32).ravel()

In [6]:
## set robot path fix collisiton 
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # /userdir
robot_path = os.path.join(ROOT, "userdir", "humanoid_research_k", "robots", "kawada_base.simple_collision.urdf")

In [7]:
log_dir = f"logs/{exp_name}"
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f"logs/{exp_name}/cfgs.pkl", "rb"))
reward_cfg["reward_scales"] = {}

In [10]:
## override
env_cfg["episode_length_s"] = 60.0
# command_cfg["lin_vel_x_range"] = [0.5, 0.5]
env_cfg['dt'] = 0.01
env_cfg['substeps'] = 10
# env_cfg["kd"] = 50
env_cfg['base_roll_noise'] = [0,0]
env_cfg['base_pitch_noise'] = [0,0]
env_cfg['termination_if_roll_greater_than'] = 150
env_cfg['termination_if_pitch_greater_than'] = 150
env_cfg['rotorInertia'] = 0.1
env_cfg["base_init_pos"] = [0.0, 0.0, 0.64]

In [11]:
reward_cfg

{'tracking_sigma': 0.25,
 'base_height_target': 0.64,
 'feet_height_target': 0.075,
 'reward_scales': {}}

In [12]:
env_cfg



{'num_actions': 12,
 'default_joint_angles': {'R_HIP_Y': 0.0,
  'R_HIP_R': 0.0,
  'R_HIP_P': -0.8,
  'R_KNEE': 1.6,
  'R_ANKLE_P': -0.8,
  'R_ANKLE_R': 0.0,
  'L_HIP_Y': 0.0,
  'L_HIP_R': 0.0,
  'L_HIP_P': -0.8,
  'L_KNEE': 1.6,
  'L_ANKLE_P': -0.8,
  'L_ANKLE_R': 0.0},
 'joint_names': ['R_HIP_Y',
  'R_HIP_R',
  'R_HIP_P',
  'R_KNEE',
  'R_ANKLE_P',
  'R_ANKLE_R',
  'L_HIP_Y',
  'L_HIP_R',
  'L_HIP_P',
  'L_KNEE',
  'L_ANKLE_P',
  'L_ANKLE_R'],
 'kp': 2000.0,
 'kd': 50.0,
 'termination_if_roll_greater_than': 150,
 'termination_if_pitch_greater_than': 150,
 'base_init_pos': [0.0, 0.0, 0.64],
 'base_init_quat': [1.0, 0.0, 0.0, 0.0],
 'episode_length_s': 60.0,
 'resampling_time_s': 4.0,
 'action_scale': 1.0,
 'simulate_action_latency': True,
 'clip_actions': 100.0,
 'dt': 0.01,
 'substeps': 10,
 'rotorInertia': 0.1,
 'base_roll_noise': [0, 0],
 'base_pitch_noise': [0, 0],
 'domain_rand': {'friction': [0.4, 1.1],
  'restitution': [0.0, 0.2],
  'kp': [1800.0, 2200.0],
  'kd': [25.0, 75.0]}}

In [13]:
env = RLEnv(
    num_envs=1,
    env_cfg=env_cfg,
    obs_cfg=obs_cfg,
    reward_cfg=reward_cfg,
    command_cfg=command_cfg,
    dt=env_cfg['dt'],
    substeps=env_cfg['substeps'],
    show_viewer=True,
    robot_urdf_path=robot_path,
)

In [14]:
runner = OnPolicyRunner(env, train_cfg, log_dir, device='cuda')
resume_path = os.path.join(log_dir, f"model_{ckpt}.pt")
runner.load(resume_path)
policy = runner.get_inference_policy(device='cuda')

obs, _ = env.reset()
cnt = 0

torques = env.sim.sbody.getTorques()

print("obs : ", obs["policy"])

# データを記録
step_data.append(cnt)
obs_data.append(_obs_vec(obs))
torque_data.append(torques.copy())

cnt += 1

--------------------------------------------------------------------------------
Resolved observation sets: 
	 policy :  ['policy']
	 critic :  ['policy']
--------------------------------------------------------------------------------
Actor MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)
obs :  tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        

In [15]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 1
Original actions :  tensor([[-0.2177, -0.1756, -0.4150, -0.2463, -1.2420, -0.0019,  0.3806,  0.5169,
          0.1421, -0.0152, -0.8218, -0.0747]], device='cuda:0')
Scaled actions :  tensor([[-0.2177, -0.1756, -0.4150, -0.2463, -1.2420, -0.0019,  0.3806,  0.5169,
          0.1421, -0.0152, -0.8218, -0.0747]], device='cuda:0')


/userdir/irsl_rl/rl_env_base.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy
/userdir/irsl_rl/rl_env_cnoid.py:103: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  self.dof_pos = torch.tensor([self.convAnglesToGenesis(sbody.angleVector())]).to(torch.float32).to(self.device)


obs :  tensor([[-4.2367e-06, -7.9481e-03, -1.6194e-06,  5.0711e-10,  2.0808e-20,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00, -8.5621e-08,
         -4.7636e-08, -1.4180e-04,  3.6466e-04, -1.9103e-04,  7.6633e-07,
          1.4362e-08,  3.0845e-08, -1.4168e-04,  3.6454e-04, -1.9103e-04,
         -3.4389e-08, -4.2811e-06, -2.3818e-06, -7.0895e-03,  1.8232e-02,
         -9.5524e-03,  3.8316e-05,  7.1809e-07,  1.5422e-06, -7.0859e-03,
          1.8227e-02, -9.5517e-03, -1.7194e-06, -2.1768e-01, -1.7564e-01,
         -4.1498e-01, -2.4629e-01, -1.2420e+00, -1.9328e-03,  3.8062e-01,
          5.1691e-01,  1.4209e-01, -1.5228e-02, -8.2180e-01, -7.4657e-02]],
       device='cuda:0')
torques: [ 3.99683909e-16 -9.12643102e-16  2.42782550e-06  7.51007967e-06
  1.66224901e-06  2.18425700e-16  2.17253195e-16 -6.61016595e-16
  2.42782550e-06  7.51007967e-06  1.66224901e-06 -7.93165041e-17]
データ収集: step 2


In [16]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 2
Original actions :  tensor([[-0.2407, -0.7349, -0.0653,  0.7013, -1.4460, -0.1905,  0.2373,  0.8060,
          0.1606,  0.1547, -0.8579,  0.0533]], device='cuda:0')
Scaled actions :  tensor([[-0.2407, -0.7349, -0.0653,  0.7013, -1.4460, -0.1905,  0.2373,  0.8060,
          0.1606,  0.1547, -0.8579,  0.0533]], device='cuda:0')
obs :  tensor([[ 2.2340e-02, -8.8129e-02, -2.8685e-01, -1.7498e-03, -3.5102e-04,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00, -3.8650e-02,
         -4.0750e-03, -8.7845e-04,  1.2010e-02, -8.5629e-02,  2.9852e-03,
          4.6567e-02,  6.3537e-03,  5.2298e-03,  1.3094e-02, -9.7825e-02,
         -1.6203e-02, -3.0669e-01, -3.9268e-02, -1.0653e-03,  8.6106e-02,
         -7.7232e-01,  2.1306e-02,  4.3912e-01,  5.2114e-02,  6.7609e-02,
          8.0966e-02, -8.7720e-01, -8.4111e-02, -2.4074e-01, -7.3488e-01,
         -6.5269e-02,  7.0128e-01, -1.4460e+00, -1.9047e-01,  2.3733e-01,
          8.0597e-01,  1.6062e-01,  1.5471e-01, -8.5787e-01,  5.3

In [17]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 3
Original actions :  tensor([[ 0.5871,  0.0648, -0.2719, -0.2291, -0.2344, -0.0208, -0.9227,  0.2908,
         -0.3624, -0.8908, -0.3034,  0.1912]], device='cuda:0')
Scaled actions :  tensor([[ 0.5871,  0.0648, -0.2719, -0.2291, -0.2344, -0.0208, -0.9227,  0.2908,
         -0.3624, -0.8908, -0.3034,  0.1912]], device='cuda:0')
obs :  tensor([[ 1.3079e-01, -3.3181e-01, -3.2226e-01, -1.0839e-02, -3.9439e-03,
         -9.9993e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -1.1385e-01,
         -2.1619e-02, -1.0512e-02,  6.1058e-02, -3.4127e-01, -5.2890e-02,
          1.1942e-01,  2.2842e-02,  2.8372e-02,  3.9410e-02, -3.3666e-01,
          1.2238e-03, -3.0990e-01, -1.4130e-01, -7.6605e-02,  3.6023e-01,
         -1.6876e+00, -2.9244e-01,  2.8845e-01,  1.1588e-01,  1.5748e-01,
          1.5438e-01, -1.1370e+00,  1.0256e-01,  5.8707e-01,  6.4840e-02,
         -2.7193e-01, -2.2913e-01, -2.3436e-01, -2.0832e-02, -9.2272e-01,
          2.9079e-01, -3.6244e-01, -8.9085e-01, -3.0342e-01,  1.9

In [18]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 4
Original actions :  tensor([[ 0.4192,  0.4626,  0.0408, -0.6592,  0.1935, -0.2931, -0.3403,  0.0193,
         -0.8554, -1.1115,  0.3880, -0.5244]], device='cuda:0')
Scaled actions :  tensor([[ 0.4192,  0.4626,  0.0408, -0.6592,  0.1935, -0.2931, -0.3403,  0.0193,
         -0.8554, -1.1115,  0.3880, -0.5244]], device='cuda:0')
obs :  tensor([[-0.4513,  0.3461, -0.3397, -0.0088,  0.0037, -1.0000,  1.0000,  0.0000,
          0.0000, -0.1226, -0.0261, -0.0445,  0.1112, -0.5704, -0.0490,  0.1268,
          0.0712,  0.0333,  0.0587, -0.4555,  0.0628,  0.1677,  0.0744, -0.2411,
          0.1647, -0.7028,  0.1291, -0.1404,  0.3163, -0.0995,  0.1153, -0.1432,
          0.3834,  0.4192,  0.4626,  0.0408, -0.6592,  0.1935, -0.2931, -0.3403,
          0.0193, -0.8554, -1.1115,  0.3880, -0.5244]], device='cuda:0')
torques: [-200.          200.         -200.         -200.          200.
  200.          200.          187.08513917 -200.         -200.
  200.           65.345065  ]
データ収集: step 5


In [19]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 5
Original actions :  tensor([[-1.3670, -1.0425,  1.1865,  0.6763,  0.0436, -0.2060,  0.7442, -0.9547,
          0.8594, -0.1596, -0.4517, -0.2453]], device='cuda:0')
Scaled actions :  tensor([[-1.3670, -1.0425,  1.1865,  0.6763,  0.0436, -0.2060,  0.7442, -0.9547,
          0.8594, -0.1596, -0.4517, -0.2453]], device='cuda:0')
obs :  tensor([[-0.6039,  0.4960, -0.0706,  0.0084,  0.0254, -0.9996,  1.0000,  0.0000,
          0.0000, -0.0455, -0.0026, -0.0763,  0.1083, -0.6068, -0.1094,  0.0487,
          0.1381,  0.0136,  0.0645, -0.3754,  0.0327,  0.5327,  0.1393, -0.0930,
         -0.1601,  0.2440, -0.4072, -0.5905,  0.3460, -0.0857, -0.0413,  0.8451,
         -0.5867, -1.3670, -1.0425,  1.1865,  0.6763,  0.0436, -0.2060,  0.7442,
         -0.9547,  0.8594, -0.1596, -0.4517, -0.2453]], device='cuda:0')
torques: [-200.         -200.         -200.         -200.          200.
 -200.          200.          200.          200.         -200.
  200.           43.65311721]
データ収集: step 6


In [20]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 6
Original actions :  tensor([[-1.1768, -0.6969,  0.1186,  0.8989, -1.0327,  0.5174,  1.3203, -0.3922,
         -0.3618, -0.3852, -1.4863,  0.4269]], device='cuda:0')
Scaled actions :  tensor([[-1.1768, -0.6969,  0.1186,  0.8989, -1.0327,  0.5174,  1.3203, -0.3922,
         -0.3618, -0.3852, -1.4863,  0.4269]], device='cuda:0')
obs :  tensor([[ 1.1798e-01, -2.0959e-01, -8.6667e-02,  1.2876e-02,  3.3910e-02,
         -9.9934e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  1.1073e-02,
         -2.5491e-03, -7.6430e-02,  9.3902e-02, -4.6854e-01, -1.5401e-01,
         -1.8126e-02,  1.7285e-01,  3.0468e-02,  3.7209e-02, -3.0704e-01,
         -1.1278e-01,  7.9588e-02, -1.0509e-01,  8.5338e-02, -9.1060e-04,
          9.8424e-01, -1.2752e-01, -1.3772e-01,  3.5270e-02,  1.9156e-01,
         -1.7721e-01, -1.0577e-01, -1.0089e+00, -1.1768e+00, -6.9692e-01,
          1.1855e-01,  8.9893e-01, -1.0327e+00,  5.1738e-01,  1.3203e+00,
         -3.9216e-01, -3.6177e-01, -3.8518e-01, -1.4863e+00,  4.2

In [21]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 7
Original actions :  tensor([[ 0.2089,  0.3386, -0.4645, -0.0763, -1.0102,  0.0604, -0.1240,  1.0010,
         -0.4888,  0.2949, -0.7873,  0.7818]], device='cuda:0')
Scaled actions :  tensor([[ 0.2089,  0.3386, -0.4645, -0.0763, -1.0102,  0.0604, -0.1240,  1.0010,
         -0.4888,  0.2949, -0.7873,  0.7818]], device='cuda:0')
obs :  tensor([[ 7.7645e-01, -2.7667e-01,  1.4457e-01,  2.2095e-03,  1.4556e-02,
         -9.9989e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -2.2955e-02,
         -5.0084e-02, -5.4355e-02,  1.0951e-01, -3.8001e-01, -7.1181e-02,
          6.6795e-03,  1.5411e-01,  5.0590e-02,  3.5754e-02, -4.1080e-01,
         -1.8036e-01, -3.7621e-01, -3.5236e-01,  1.1005e-01,  1.4236e-01,
         -2.1239e-04,  8.5668e-01,  3.6433e-01, -2.1547e-01,  5.5212e-02,
          5.7227e-03, -9.0279e-01,  1.7042e-01,  2.0890e-01,  3.3862e-01,
         -4.6454e-01, -7.6303e-02, -1.0102e+00,  6.0397e-02, -1.2396e-01,
          1.0010e+00, -4.8875e-01,  2.9490e-01, -7.8731e-01,  7.8

In [22]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 8
Original actions :  tensor([[ 0.7232,  0.7825, -0.4037, -1.3512, -0.1683, -0.9952, -0.2269,  0.9104,
          0.1531,  0.0964,  0.0290, -0.5265]], device='cuda:0')
Scaled actions :  tensor([[ 0.7232,  0.7825, -0.4037, -1.3512, -0.1683, -0.9952, -0.2269,  0.9104,
          0.1531,  0.0964,  0.0290, -0.5265]], device='cuda:0')
obs :  tensor([[ 7.8054e-02,  3.8052e-01,  1.4213e-01,  5.5744e-03, -1.0733e-03,
         -9.9998e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -4.9985e-02,
         -9.4539e-02, -4.9615e-02,  1.2014e-01, -4.8460e-01,  6.6448e-02,
          3.4108e-02,  1.4086e-01,  2.5199e-02,  7.2326e-02, -5.4889e-01,
         -4.2128e-02,  5.4504e-02, -1.0371e-01, -4.0076e-02, -1.5820e-02,
         -9.5141e-01,  2.6681e-01, -4.9629e-02,  5.3654e-02, -2.8044e-01,
          3.2103e-01, -5.1840e-01,  1.1170e+00,  7.2321e-01,  7.8249e-01,
         -4.0370e-01, -1.3512e+00, -1.6834e-01, -9.9517e-01, -2.2686e-01,
          9.1040e-01,  1.5312e-01,  9.6394e-02,  2.8982e-02, -5.2

In [23]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 9
Original actions :  tensor([[-1.1603, -0.1292,  0.9004, -0.1689,  0.2320,  0.2992,  0.6083, -0.8296,
          1.3353, -0.1213, -0.0907, -0.7120]], device='cuda:0')
Scaled actions :  tensor([[-1.1603, -0.1292,  0.9004, -0.1689,  0.2320,  0.2992,  0.6083, -0.8296,
          1.3353, -0.1213, -0.0907, -0.7120]], device='cuda:0')
obs :  tensor([[-0.4851,  0.4512, -0.3007,  0.0223,  0.0080, -0.9997,  1.0000,  0.0000,
          0.0000,  0.0102, -0.0908, -0.0579,  0.0987, -0.5665,  0.0117, -0.0258,
          0.1841, -0.0060,  0.1092, -0.5480,  0.0702,  0.5097,  0.1185, -0.0482,
         -0.1828,  0.0336, -0.7165, -0.4139,  0.3521, -0.0501,  0.0724,  0.4324,
          0.1098, -1.1603, -0.1292,  0.9004, -0.1689,  0.2320,  0.2992,  0.6083,
         -0.8296,  1.3353, -0.1213, -0.0907, -0.7120]], device='cuda:0')
torques: [  -7.49448865  200.          200.         -115.55079567  200.
 -200.          200.          200.         -200.         -200.
  200.         -200.        ]
データ収集: step 10

In [24]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 10
Original actions :  tensor([[-1.1645, -0.8285,  0.3333,  0.7697, -0.8357,  0.8414,  1.2808, -0.9714,
         -0.5776, -0.1794, -0.8996,  0.2410]], device='cuda:0')
Scaled actions :  tensor([[-1.1645, -0.8285,  0.3333,  0.7697, -0.8357,  0.8414,  1.2808, -0.9714,
         -0.5776, -0.1794, -0.8996,  0.2410]], device='cuda:0')
obs :  tensor([[-3.1519e-02, -2.2319e-01, -3.3787e-01,  2.5321e-02,  1.7331e-02,
         -9.9953e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  5.8010e-02,
         -8.0536e-02, -5.7879e-02,  7.2369e-02, -4.7306e-01,  4.1073e-03,
         -6.4243e-02,  2.2841e-01,  2.3319e-02,  8.5286e-02, -4.0310e-01,
         -1.1556e-02,  2.6685e-02,  2.2011e-04,  4.0942e-02, -9.0051e-02,
          8.1712e-01,  4.9714e-01, -1.4619e-02,  1.1666e-01,  3.0063e-01,
         -2.7824e-01,  6.7884e-01, -8.3317e-01, -1.1645e+00, -8.2845e-01,
          3.3333e-01,  7.6965e-01, -8.3567e-01,  8.4142e-01,  1.2808e+00,
         -9.7138e-01, -5.7758e-01, -1.7938e-01, -8.9964e-01,  2.

In [25]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 11
Original actions :  tensor([[ 0.3807,  0.2040, -0.2869, -0.2831, -0.9447, -0.2317, -0.4045,  1.0134,
         -1.0212,  0.9077, -1.2837,  0.8268]], device='cuda:0')
Scaled actions :  tensor([[ 0.3807,  0.2040, -0.2869, -0.2831, -0.9447, -0.2317, -0.4045,  1.0134,
         -1.0212,  0.9077, -1.2837,  0.8268]], device='cuda:0')
obs :  tensor([[ 0.6639, -0.3092, -0.1147,  0.0155,  0.0040, -0.9999,  1.0000,  0.0000,
          0.0000,  0.0140, -0.1070, -0.0484,  0.0701, -0.4178,  0.1855, -0.0068,
          0.2192,  0.0642,  0.0284, -0.3699, -0.0694, -0.4120, -0.2507,  0.0579,
          0.0506, -0.1660,  1.2361,  0.4555, -0.1341,  0.0973, -0.0436, -0.1622,
          0.0420,  0.3807,  0.2040, -0.2869, -0.2831, -0.9447, -0.2317, -0.4045,
          1.0134, -1.0212,  0.9077, -1.2837,  0.8268]], device='cuda:0')
torques: [ 200.         -200.         -200.          -89.35054689 -200.
  200.         -200.         -200.          200.          200.
 -200.          200.        ]
データ収集: step 1

In [39]:
# 既存のforループを置き換え
num_steps = 100
for i in range(num_steps):
    with torch.no_grad():
        actions = policy(obs)
        
        # アクションスケーリング
        scaled_actions = actions * action_scale
        obs, rews, dones, infos = env.step(scaled_actions)  # スケール済みを使用
        torques = env.sim.sbody.getTorques()
        
        # データを記録
        step_data.append(cnt)
        obs_data.append(_obs_vec(obs))
        torque_data.append(torques.copy())
        
        # デバッグ表示（最初の数ステップのみ）
        if i < 3:
            print(f"Step {i}: Original action max={actions.max():.3f}, "
                  f"Scaled action max={scaled_actions.max():.3f}")
        
        if i % 20 == 0:
            print(f"Step {i+1}/{num_steps}, Total steps: {cnt}")
            print("steps:",cnt)
            print("actions :",scaled_actions)
            print("target_dof_pos:",env.target_dof_pos)
        
        cnt += 1

print(f"データ収集完了: {num_steps} steps collected with action_scale={action_scale}")

Step 0: Original action max=1.125, Scaled action max=1.125
Step 1/100, Total steps: 1312
steps: 1312
actions : tensor([[-0.8481, -0.6517, -0.5809,  0.7498, -0.2123,  0.2921,  1.1252, -1.4991,
         -0.7197,  0.7737, -0.2366, -0.1794]], device='cuda:0')
target_dof_pos: tensor([[ 1.1097,  0.9885, -0.3088,  1.5903, -1.8199, -0.4635,  0.6994,  0.8837,
         -1.2033,  1.7233, -1.2482,  0.0351]], device='cuda:0')
Step 1: Original action max=0.885, Scaled action max=0.885
Step 2: Original action max=1.795, Scaled action max=1.795
Step 21/100, Total steps: 1332
steps: 1332
actions : tensor([[ 0.6221, -0.8684, -0.8072, -0.0035, -1.1881, -1.1157,  0.1187, -0.3166,
         -0.5214, -0.7554, -0.6389,  0.0961]], device='cuda:0')
target_dof_pos: tensor([[ 0.0847,  0.6576,  0.2709,  1.4552, -1.9916,  0.0821,  1.4946,  1.2713,
         -0.0073,  1.6588, -0.5287,  0.5434]], device='cuda:0')
Step 41/100, Total steps: 1352
steps: 1352
actions : tensor([[-1.2403, -0.7344,  0.0028, -0.7895, -0.3356,

In [40]:
# for i in range(500):
#     with torch.no_grad():
#         actions = policy(obs)
#         scaled_actions = actions * action_scale
#         obs, rews, dones, infos = env.step(scaled_actions)

In [41]:
env.sim.stop()

In [31]:
env.reset()
cnt = 0

In [61]:
# 最もシンプルな保存方法
def save_simple_csv():
    if not step_data:
        print("データがありません")
        return
    
    # 基本的な辞書形式でデータを整理
    data_dict = {'step': step_data}
    
    # # Actionデータ
    # action_array = np.array(action_data)
    # for i in range(action_array.shape[1]):
    #     data_dict[f'action_{i}'] = action_array[:, i]
    
    # Observationデータ
    obs_array = np.array(obs_data)
    for i in range(obs_array.shape[1]):
        data_dict[f'obs_{i}'] = obs_array[:, i]
    
    # Torqueデータ
    torque_array = np.array(torque_data)
    for i in range(torque_array.shape[1]):
        data_dict[f'torque_{i}'] = torque_array[:, i]
    
    # DataFrameを作成して保存
    df = pd.DataFrame(data_dict)
    csv_filename = f'obs_data/cnoid_{exp_name}_ckpt{ckpt}_scale{action_scale}_rotorInertia0.1.csv'
    df.to_csv(csv_filename, index=False)
    
    print(f"シンプル版を保存: {csv_filename}")
    print(f"データ形状: {df.shape}")
    
    return df

# シンプル版を実行
df_simple = save_simple_csv()

シンプル版を保存: obs_data/cnoid_friction-walking-terrain2-kp2000kd50-kpkdrand-14_ckpt100_scale1.0_rotorInertia0.1.csv
データ形状: (362, 58)
